# LegaLoom-Env: Adversarial Benchmark

Runs **20 hand-curated adversarial TDS scenarios** against:
1. **Baseline** — untrained Qwen2.5-3B-Instruct
2. **Trained** — same model + LoRA from `LegaLoom_FullCurriculum.ipynb`
3. **Frontier models** (optional, requires API keys): GPT-4o-mini, Claude Sonnet 4, Gemini 2.5 Pro

Output: leaderboard chart + `adversarial_results.json` showing per-case scores and aggregate metrics by failure-mode category.

**Why this matters:** the 20 cases are held-out, hand-written failure-mode probes that target known LLM weaknesses on Indian TDS compliance (inoperative-PAN low-base-rate, FY 2025-26 new sections, threshold-boundary off-by-one, mixed goods+services positional confusion). They never appear in training. A model's score here measures genuine compliance reasoning, not memorization.

In [ ]:
# Cell 1: Detect platform + install
import os
ON_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')
print(f'Platform: {"Colab" if ON_COLAB else "HF Space / Local"}')

if ON_COLAB:
    !pip install unsloth 'trl>=0.12.0' openenv-core==0.2.3 fastapi pydantic \
        openai anthropic google-generativeai pyyaml matplotlib --quiet
else:
    !pip install -q 'transformers>=4.46,<4.50' 'peft>=0.13' bitsandbytes \
        accelerate openai anthropic google-generativeai pyyaml matplotlib \
        openenv-core==0.2.3 fastapi pydantic
print('✓ Installed')

In [ ]:
# Cell 2: Clone repo
import sys, os, subprocess
if ON_COLAB:
    WORK_DIR = '/content/legaloom_env'
else:
    WORK_DIR = '/data/legaloom-env' if os.path.exists('/data') and os.access('/data', os.W_OK) else os.path.expanduser('~/legaloom-env')

if not os.path.exists(WORK_DIR):
    subprocess.run(['git', 'clone',
                    'https://huggingface.co/spaces/aarav0202/legaloom-env',
                    WORK_DIR], check=True)
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print(f'✓ Working dir: {os.getcwd()}')

In [ ]:
# Cell 3: Load adversarial test set + define unified prompt
import json
from server.adversarial_cases import (
    ADVERSARIAL_CASES, score_adversarial, get_categories
)

print(f'Loaded {len(ADVERSARIAL_CASES)} adversarial cases')
print(f'Categories: {get_categories()}')
print()

ADVERSARIAL_PROMPT = '''You are a TDS compliance agent for Indian businesses (FY 2025-26).

Given the invoice below, determine the correct TDS deduction.

Output ONLY a JSON object with these keys:
  - tds_amount_inr: float, the rupee amount to deduct
  - section: string, e.g. "194J", "194C", "206AA", or "no_tds" if below threshold
  - rate_percent: float, e.g. 10.0, 2.0, 20.0, or 0.0 for no_tds
  - no_tds: "true" if claiming below threshold, otherwise omit

Key rules to remember:
  - INOPERATIVE PAN → Section 206AA, rate 20% flat (overrides any other section)
  - 194J Professional (individual/LLP) = 10%; 194J Technical (Pvt Ltd) = 2%
  - 194I Building rent = 10%; 194I Machinery = 2%
  - 194T (NEW FY25-26) partner drawings = 10%, threshold ₹20,000
  - 194Q goods purchase = 0.1% (only if buyer turnover > ₹10cr in prior FY)
  - Mixed goods+services: TDS applies ONLY to the services portion
  - Threshold check: TDS only if (cumulative_ytd + current_amount) crosses the section threshold

INVOICE:
{invoice_text}

VENDOR PAN STATUS: {pan_status}
CUMULATIVE YTD PAYMENTS TO THIS VENDOR: ₹{cumulative_ytd}

Respond with JSON only.'''

def render_prompt(case):
    return ADVERSARIAL_PROMPT.format(
        invoice_text=case['invoice_text'],
        pan_status=case.get('pan_status', 'operative'),
        cumulative_ytd=case.get('cumulative_ytd', 0.0),
    )

print('Sample prompt:')
print(render_prompt(ADVERSARIAL_CASES[0])[:500] + '...')

In [ ]:
# Cell 4: JSON extraction helper (line-anchored fence strip — same as train_grpo.py)
import re

def extract_json(text: str):
    cleaned = re.sub(r'^[ \t]*```(?:json|JSON)?[ \t]*\n?', '', text, flags=re.MULTILINE)
    cleaned = re.sub(r'\n[ \t]*```[ \t]*$', '', cleaned, flags=re.MULTILINE)
    match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)?\}', cleaned, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except Exception:
        return None

# Smoke test
test = '```json\n{"section": "194J", "rate_percent": 10.0, "tds_amount_inr": 5000}\n```'
print('Extracted:', extract_json(test))

In [ ]:
# Cell 5: Load Qwen2.5-3B (baseline) + optionally trained LoRA
# Skip this cell if you only want to benchmark frontier APIs.
import torch

qwen_available = True
try:
    if ON_COLAB:
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name='unsloth/Qwen2.5-3B-Instruct',
            max_seq_length=2048, dtype=None, load_in_4bit=True,
        )
        # Add LoRA scaffolding so trained adapter (if any) can be loaded later
        model = FastLanguageModel.get_peft_model(
            model, r=16,
            target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
            lora_alpha=16, lora_dropout=0, bias='none',
            use_gradient_checkpointing='unsloth', random_state=42,
        )
        FastLanguageModel.for_inference(model)
    else:
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                                  bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
        tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            'Qwen/Qwen2.5-3B-Instruct',
            quantization_config=bnb, device_map='auto', torch_dtype=torch.bfloat16,
        )
        model.eval()
    print(f'✓ Qwen2.5-3B loaded. dtype={next(model.parameters()).dtype}')
except Exception as e:
    print(f'⚠ Could not load Qwen2.5-3B: {e}')
    print('  Continuing with frontier-API benchmarks only.')
    qwen_available = False

In [ ]:
# Cell 6: Run Qwen baseline (untrained) on all 20 cases
import torch

def query_local_model(prompt, max_new_tokens=256, temperature=0.1):
    msgs = [{'role': 'user', 'content': prompt}]
    chat = tokenizer.apply_chat_template(
        msgs, return_tensors='pt', add_generation_prompt=True, return_dict=True
    )
    input_ids = chat['input_ids'].to(model.device)
    attention_mask = chat['attention_mask'].to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids, attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    return text

baseline_submissions = {}
if qwen_available:
    print('Running Qwen-3B baseline on 20 adversarial cases...')
    for case in ADVERSARIAL_CASES:
        prompt = render_prompt(case)
        try:
            response = query_local_model(prompt)
            sub = extract_json(response) or {}
        except Exception as e:
            sub = {}
        baseline_submissions[case['id']] = sub
        result = score_adversarial(sub, case)
        print(f"  {case['id']}: score={result['score']:.2f}")
    print(f'✓ Baseline complete')
else:
    print('Qwen unavailable; skipping baseline')

In [ ]:
# Cell 7: Run Qwen TRAINED on all 20 cases (uses the LoRA adapter loaded above)
# If you trained in this same session via LegaLoom_FullCurriculum.ipynb,
# the model variable still holds the trained adapter weights.
# Otherwise, this measures the same baseline twice — that's expected if you skipped training.
trained_submissions = {}
if qwen_available:
    print('Running Qwen-3B trained on 20 adversarial cases...')
    for case in ADVERSARIAL_CASES:
        prompt = render_prompt(case)
        try:
            response = query_local_model(prompt)
            sub = extract_json(response) or {}
        except Exception as e:
            sub = {}
        trained_submissions[case['id']] = sub
        result = score_adversarial(sub, case)
        print(f"  {case['id']}: score={result['score']:.2f}")
    print(f'✓ Trained complete')
else:
    print('Qwen unavailable; skipping')

In [ ]:
# Cell 8: Frontier model benchmarks (optional — set API keys as env vars)
# OPENAI_API_KEY, ANTHROPIC_API_KEY, GOOGLE_API_KEY
import os, time

frontier_results = {}  # provider -> {case_id -> submission_dict}

def run_openai(prompt, model_name='gpt-4o-mini'):
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
    resp = client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=256, temperature=0.0,
    )
    return resp.choices[0].message.content

def run_anthropic(prompt, model_name='claude-sonnet-4-5'):
    import anthropic
    client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
    resp = client.messages.create(
        model=model_name, max_tokens=256, temperature=0.0,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return resp.content[0].text

def run_gemini(prompt, model_name='gemini-2.5-pro'):
    import google.generativeai as genai
    genai.configure(api_key=os.environ['GOOGLE_API_KEY'])
    m = genai.GenerativeModel(model_name)
    resp = m.generate_content(prompt, generation_config={'temperature': 0.0, 'max_output_tokens': 256})
    return resp.text

providers = []
if os.environ.get('OPENAI_API_KEY'):
    providers.append(('GPT-4o-mini', run_openai))
if os.environ.get('ANTHROPIC_API_KEY'):
    providers.append(('Claude Sonnet 4.5', run_anthropic))
if os.environ.get('GOOGLE_API_KEY'):
    providers.append(('Gemini 2.5 Pro', run_gemini))

if not providers:
    print('No frontier API keys found in environment.')
    print('Set OPENAI_API_KEY, ANTHROPIC_API_KEY, or GOOGLE_API_KEY to run frontier benchmarks.')
else:
    for name, fn in providers:
        print(f'\n=== {name} ===')
        frontier_results[name] = {}
        for case in ADVERSARIAL_CASES:
            prompt = render_prompt(case)
            try:
                response = fn(prompt)
                sub = extract_json(response) or {}
            except Exception as e:
                print(f'  {case["id"]}: ERROR {e}')
                sub = {}
            frontier_results[name][case['id']] = sub
            result = score_adversarial(sub, case)
            print(f"  {case['id']}: score={result['score']:.2f}")
            time.sleep(0.3)  # rate-limit cushion

In [ ]:
# Cell 9: Aggregate scores per model
def aggregate(submissions):
    if not submissions:
        return {'overall_mean': 0.0, 'by_category': {}, 'per_case': {}}
    per_case, by_cat = {}, {}
    for case in ADVERSARIAL_CASES:
        sub = submissions.get(case['id'], {})
        result = score_adversarial(sub, case)
        per_case[case['id']] = result['score']
        cat = case['category']
        by_cat.setdefault(cat, []).append(result['score'])
    by_cat_avg = {k: sum(v)/len(v) for k, v in by_cat.items()}
    return {
        'overall_mean': sum(per_case.values()) / len(per_case),
        'by_category': by_cat_avg,
        'per_case': per_case,
    }

all_results = {}
if baseline_submissions:
    all_results['Qwen2.5-3B (baseline)'] = aggregate(baseline_submissions)
if trained_submissions:
    all_results['Qwen2.5-3B (trained)'] = aggregate(trained_submissions)
for name, subs in frontier_results.items():
    all_results[name] = aggregate(subs)

# Print leaderboard
print('=' * 50)
print('ADVERSARIAL BENCHMARK LEADERBOARD')
print('=' * 50)
ranked = sorted(all_results.items(), key=lambda kv: -kv[1]['overall_mean'])
for i, (name, agg) in enumerate(ranked, 1):
    print(f"{i}. {name:32s}  {agg['overall_mean']:.3f}")

# Save
with open('adversarial_results.json', 'w') as f:
    json.dump({
        'leaderboard': [(n, a['overall_mean']) for n, a in ranked],
        'by_model': all_results,
        'baseline_submissions': baseline_submissions,
        'trained_submissions': trained_submissions,
        'frontier_submissions': frontier_results,
    }, f, indent=2, default=str)
print('\n✓ adversarial_results.json saved')

In [ ]:
# Cell 10: Bar chart — overall + per-category leaderboard
import matplotlib.pyplot as plt

if not all_results:
    print('No results to plot. Run Cells 6, 7, or 8 first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: overall leaderboard
    names = [n for n, _ in ranked]
    scores = [a['overall_mean'] for _, a in ranked]
    colors = ['#2a9d8f' if 'trained' in n else '#e76f51' if 'baseline' in n else '#264653'
              for n in names]
    ax = axes[0]
    bars = ax.barh(names, scores, color=colors)
    for b, s in zip(bars, scores):
        ax.text(b.get_width() + 0.01, b.get_y() + b.get_height()/2,
                f'{s:.3f}', va='center', fontsize=10)
    ax.set_xlim(0, 1.0)
    ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.4, label='Success threshold')
    ax.set_xlabel('Mean Score (20 adversarial cases)')
    ax.set_title('Adversarial Benchmark Leaderboard')
    ax.legend()
    ax.invert_yaxis()

    # Right: per-category for the best 3 (or all if ≤3)
    top_models = ranked[:3]
    cats = sorted({c for _, a in top_models for c in a['by_category']})
    ax2 = axes[1]
    width = 0.8 / len(top_models)
    x_pos = list(range(len(cats)))
    for j, (mname, agg) in enumerate(top_models):
        vals = [agg['by_category'].get(c, 0) for c in cats]
        ax2.bar([p + j*width for p in x_pos], vals, width, label=mname[:18])
    ax2.set_xticks([p + width*(len(top_models)-1)/2 for p in x_pos])
    ax2.set_xticklabels([c.replace('_', '\n') for c in cats], rotation=0, fontsize=8)
    ax2.set_ylabel('Mean Score')
    ax2.set_title('Top 3 Models — Score by Failure-Mode Category')
    ax2.set_ylim(0, 1.0)
    ax2.legend(fontsize=8)
    ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.savefig('adversarial_leaderboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ adversarial_leaderboard.png saved')